In [1]:
import os
os.environ["NPU_VISIBLE_DEVICES"]="0"
os.environ["ASCEND_RT_VISIBLE_DEVICES"]="0"
from typing import Dict, List, Any

import torch
import torch_npu

from datasets import load_dataset
from transformers import LlamaForCausalLM, AutoTokenizer

/home/lihz/miniconda3/envs/workspace/lib/python3.10/site-packages/torch_npu/utils/path_manager.py:82: UserWarning: Warning: The /usr/local/Ascend/ascend-toolkit/latest owner does not match the current user.
  warnings.warn(f"Warning: The {path} owner does not match the current user.")
/home/lihz/miniconda3/envs/workspace/lib/python3.10/site-packages/torch_npu/utils/path_manager.py:82: UserWarning: Warning: The /usr/local/Ascend/ascend-toolkit/8.0.RC2/aarch64-linux/ascend_toolkit_install.info owner does not match the current user.
  warnings.warn(f"Warning: The {path} owner does not match the current user.")


In [2]:
tokenizer = AutoTokenizer.from_pretrained("/data/pretrained-models/meta/Llama-3.2-1B-Instruct")
dataset = load_dataset("/data/datasets/Magicoder-Evol-Instruct-110K")["train"]
model = LlamaForCausalLM.from_pretrained(
    "/data/pretrained-models/meta/Llama-3.2-1B-Instruct",
    torch_dtype=torch.bfloat16,
    device_map="npu:0")

In [3]:
def preprocess(example:Dict[str, str])->Dict[str, str]:
    return {
        "inst": f"<|begin_of_text|><|start_header_id|>user<|end_header_id|>\n\n{example['instruction']}<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n\n",
        "resp": f"{example['response']}<|eot_id|>"
    }
def postprocess(example:Dict[str, str])->Dict[str, List[int]]:
    inst = tokenizer.encode(example["inst"], add_special_tokens=False)
    resp = tokenizer.encode(example["resp"], add_special_tokens=False)
    return {
        "input_ids": inst + resp,
        "labels": [-100] * len(inst) + resp
    }

In [4]:
tag = 10
dataset = dataset.map(
    preprocess,
    num_proc=4,
    remove_columns=dataset.column_names,
    # load_from_cache_file=False,
)
print(dataset[tag]['inst']+dataset[tag]['resp'])
dataset = dataset.map(
    postprocess,
    num_proc=4,
    remove_columns=dataset.column_names,
    # load_from_cache_file=False,
)
print(tokenizer.decode(dataset[tag]['input_ids']))

<|begin_of_text|><|start_header_id|>user<|end_header_id|>

Formulate a programming blueprint to integrate an advanced GPT-3 model, using the PyTorch library, for the purpose of undertaking text translation tasks on a comprehensive text corpus within a framework that utilises not just a single but multi-GPU setup and optimizes the efficient use of grid computing.<|eot_id|><|start_header_id|>assistant<|end_header_id|>

Designing such a complex blueprint would involve multiple modules and tasks, including working with GPT-3, PyTorch, multi-GPU computing, and efficient utilization of grid computing. Below is a simplified blueprint:

1. **Import Libraries** 
Begin by importing necessary modules and libraries from PyTorch such as torch, nn, optim, and from the transformers library import GPT3LMHeadModel, GPT2Tokenizer.

2. **Prepare your Data**
Next, retrieve your corpus and preprocess it to be suitable for GPT-3. The transformer model requires a specific format of data. 

3. **Creating Mode

In [5]:
module_dicts = {}
for name, module in model.named_modules():
    # print(name)
    module_dicts[name] = module
print(module_dicts.keys())
# module_dicts[name]

dict_keys(['', 'model', 'model.embed_tokens', 'model.layers', 'model.layers.0', 'model.layers.0.self_attn', 'model.layers.0.self_attn.q_proj', 'model.layers.0.self_attn.k_proj', 'model.layers.0.self_attn.v_proj', 'model.layers.0.self_attn.o_proj', 'model.layers.0.self_attn.rotary_emb', 'model.layers.0.mlp', 'model.layers.0.mlp.gate_proj', 'model.layers.0.mlp.up_proj', 'model.layers.0.mlp.down_proj', 'model.layers.0.mlp.act_fn', 'model.layers.0.input_layernorm', 'model.layers.0.post_attention_layernorm', 'model.layers.1', 'model.layers.1.self_attn', 'model.layers.1.self_attn.q_proj', 'model.layers.1.self_attn.k_proj', 'model.layers.1.self_attn.v_proj', 'model.layers.1.self_attn.o_proj', 'model.layers.1.self_attn.rotary_emb', 'model.layers.1.mlp', 'model.layers.1.mlp.gate_proj', 'model.layers.1.mlp.up_proj', 'model.layers.1.mlp.down_proj', 'model.layers.1.mlp.act_fn', 'model.layers.1.input_layernorm', 'model.layers.1.post_attention_layernorm', 'model.layers.2', 'model.layers.2.self_attn'

In [6]:

def hook_fn(
        module: torch.nn.Module,
        module_inputs: Any,
        module_outputs: Any,
):
    print(type(module), type(module_inputs), type(module_outputs))
    print(len(module_inputs))
    print(module_inputs[0].shape)
# handle = module_dicts["model.layers.4.mlp.act_fn"].register_forward_hook(
#     hook_fn
# )
handle = module_dicts["model.layers.7.mlp.gate_proj"].register_forward_hook(
    hook_fn
)

In [7]:
with torch.no_grad():
    input_kwargs = {
        "input_ids": torch.tensor([dataset[tag]["input_ids"]], device=model.device),
        "labels": torch.tensor([dataset[tag]["labels"]], device=model.device),
        "use_cache": False,
    }
    ift_outputs = model(**input_kwargs)

<class 'torch.nn.modules.linear.Linear'> <class 'tuple'> <class 'torch.Tensor'>
1
torch.Size([1, 544, 2048])
